# WBC Quality Feature Extraction and Severity Analysis

Source notebook extracted from the original project notebook. It processes seven separate pristine/degraded Phase 1 and Phase 2 datasets. The source Parquet dataset must be available under /content/wbcbench2026. Run the cells in order. Processing is not started automatically.

## Feature extraction function

In [ ]:
import cv2
import numpy as np
import pyarrow.parquet as pq

def extract_quality_features(image_bytes):

    image = cv2.imdecode(
        np.frombuffer(image_bytes, np.uint8),
        cv2.IMREAD_COLOR
    )

    if image is None:
        return None

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    laplacian_variance = cv2.Laplacian(
        gray, cv2.CV_64F
    ).var()

    median = cv2.medianBlur(gray, 3)

    median_residual = np.mean(
        np.abs(gray.astype(np.float32) -
               median.astype(np.float32))
    )

    brightness = np.mean(gray)
    contrast = np.std(gray)

    extreme_percentage = (
        np.sum((gray <= 5) | (gray >= 250))
        / gray.size
    ) * 100

    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    rgb_variation = np.mean(
        np.std(rgb.astype(np.float32), axis=(0, 1))
    )

    edges = cv2.Canny(gray, 100, 200)

    edge_density = (
        np.count_nonzero(edges) / edges.size
    ) * 100

    return {
        "laplacian_variance": laplacian_variance,
        "median_filter_residual": median_residual,
        "brightness": brightness,
        "contrast": contrast,
        "extreme_pixel_percentage": extreme_percentage,
        "rgb_colour_variation": rgb_variation,
        "edge_density": edge_density
    }


test_file = "/content/wbcbench2026/pristine/phase1_train-00000-of-00001.parquet"

parquet = pq.ParquetFile(test_file)
batch = next(parquet.iter_batches(batch_size=1, columns=["bytes", "image_id"]))

image_bytes = batch.column("bytes")[0].as_py()
image_id = batch.column("image_id")[0].as_py()

features = extract_quality_features(image_bytes)

print("Image ID:", image_id)
print("Image decoded successfully:", features is not None)
print("Extracted features:", features)

## Seven-dataset feature extraction

In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

BASE = "/content/wbcbench2026"
OUTPUT = "/content/WBC_Quality_Feature_CSVs"

os.makedirs(OUTPUT, exist_ok=True)

datasets = {
    "phase1_train_pristine": [
        "pristine/phase1_train-00000-of-00001.parquet"
    ],
    "phase2_train_pristine": [
        "pristine/phase2_train-00000-of-00001.parquet"
    ],
    "phase2_eval_pristine": [
        "pristine/phase2_eval-00000-of-00001.parquet"
    ],
    "phase2_test_pristine": [
        "pristine/phase2_test-00000-of-00001.parquet"
    ],
    "phase2_train_degraded": [
        f"degraded/phase2_train-0000{i}-of-00004.parquet"
        for i in range(4)
    ],
    "phase2_eval_degraded": [
        "degraded/phase2_eval-00000-of-00001.parquet"
    ],
    "phase2_test_degraded": [
        f"degraded/phase2_test-0000{i}-of-00002.parquet"
        for i in range(2)
    ]
}

feature_columns = [
    "laplacian_variance",
    "median_filter_residual",
    "brightness",
    "contrast",
    "extreme_pixel_percentage",
    "rgb_colour_variation",
    "edge_density"
]

metadata_columns = [
    "image_id",
    "split",
    "class_label",
    "severity_label"
]

expected_columns = metadata_columns + feature_columns

batch_size = 32
failure_records = []

for dataset_name, source_files in datasets.items():

    output_file = os.path.join(
        OUTPUT,
        dataset_name + "_quality_features.csv"
    )

    if os.path.exists(output_file):
        os.remove(output_file)

    total_processed = 0
    total_failed = 0

    print("\n" + "=" * 65)
    print("Processing:", dataset_name)

    for source_file in source_files:

        parquet_path = os.path.join(BASE, source_file)
        parquet = pq.ParquetFile(parquet_path)

        total_batches = (
            parquet.metadata.num_rows + batch_size - 1
        ) // batch_size

        progress = tqdm(
            parquet.iter_batches(
                batch_size=batch_size,
                columns=[
                    "image",
                    "image_id",
                    "label",
                    "wbcbench_split",
                    "severity"
                ]
            ),
            total=total_batches,
            desc=source_file.split("/")[-1]
        )

        for batch in progress:

            image_index = batch.schema.get_field_index("image")

            image_struct = batch.column(image_index)

            image_bytes_list = (
                image_struct.field("bytes").to_pylist()
            )

            image_ids = batch.column(
                batch.schema.get_field_index("image_id")
            ).to_pylist()

            labels = batch.column(
                batch.schema.get_field_index("label")
            ).to_pylist()

            splits = batch.column(
                batch.schema.get_field_index("wbcbench_split")
            ).to_pylist()

            severities = batch.column(
                batch.schema.get_field_index("severity")
            ).to_pylist()

            records = []

            for image_bytes, image_id, label, split, severity in zip(
                image_bytes_list,
                image_ids,
                labels,
                splits,
                severities
            ):

                features = (
                    extract_quality_features(image_bytes)
                    if image_bytes is not None
                    else None
                )

                record = {
                    "image_id": image_id,
                    "split": split,
                    "class_label": label,
                    "severity_label": severity
                }

                if features is None:

                    total_failed += 1

                    failure_records.append({
                        "dataset": dataset_name,
                        "source_file": source_file,
                        "image_id": image_id
                    })

                    record.update({
                        column: np.nan
                        for column in feature_columns
                    })

                else:
                    record.update(features)

                records.append(record)

            df = pd.DataFrame(
                records,
                columns=expected_columns
            )

            df.to_csv(
                output_file,
                mode="a",
                header=not os.path.exists(output_file + ".written"),
                index=False
            )

            if not os.path.exists(output_file + ".written"):
                open(output_file + ".written", "w").close()

            total_processed += len(df)

    marker = output_file + ".written"

    if os.path.exists(marker):
        os.remove(marker)

    print("Rows processed:", total_processed)
    print("Failed image extractions:", total_failed)
    print("CSV saved:", output_file)

    if total_processed == 0:
        raise RuntimeError(
            f"No records were processed for {dataset_name}"
        )

if failure_records:
    pd.DataFrame(failure_records).to_csv(
        os.path.join(OUTPUT, "failed_image_extractions.csv"),
        index=False
    )

print("\n" + "=" * 65)
print("ALL SEVEN CSV GENERATION TASKS FINISHED")
print("Output folder:", OUTPUT)
print("Total failed extractions:", len(failure_records))

## CSV validation

In [ ]:

import os
import pandas as pd

OUTPUT = "/content/WBC_Quality_Feature_CSVs"

expected_rows = {
    "phase1_train_pristine_quality_features.csv": 8288,
    "phase2_train_pristine_quality_features.csv": 24897,
    "phase2_eval_pristine_quality_features.csv": 5350,
    "phase2_test_pristine_quality_features.csv": 16467,
    "phase2_train_degraded_quality_features.csv": 24897,
    "phase2_eval_degraded_quality_features.csv": 5350,
    "phase2_test_degraded_quality_features.csv": 16467
}

expected_columns = [
    "image_id",
    "split",
    "class_label",
    "severity_label",
    "laplacian_variance",
    "median_filter_residual",
    "brightness",
    "contrast",
    "extreme_pixel_percentage",
    "rgb_colour_variation",
    "edge_density"
]

all_passed = True

for filename, expected_count in expected_rows.items():

    path = os.path.join(OUTPUT, filename)
    df = pd.read_csv(path)

    row_check = len(df) == expected_count
    column_check = list(df.columns) == expected_columns
    duplicate_count = df["image_id"].duplicated().sum()
    missing_count = df.isnull().sum().sum()

    print("\n" + "=" * 60)
    print("FILE:", filename)
    print("Rows:", len(df), "| Expected:", expected_count)
    print("Correct row count:", row_check)
    print("Correct columns:", column_check)
    print("Duplicate image IDs:", duplicate_count)
    print("Total missing values:", missing_count)

    print("\nSeverity counts:")
    print(df["severity_label"].value_counts(dropna=False))

    passed = (
        row_check
        and column_check
        and duplicate_count == 0
        and missing_count == 0
    )

    print("\nValidation:", "PASS" if passed else "FAIL")

    if not passed:
        all_passed = False

print("\n" + "=" * 60)
print("FINAL VALIDATION:",
      "ALL SEVEN CSV FILES PASSED" if all_passed
      else "SOME FILES NEED INVESTIGATION")


## Severity analysis and plots

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile

INPUT = "/content/WBC_Quality_Feature_CSVs"
OUTPUT = "/content/WBC_Severity_Analysis"

os.makedirs(OUTPUT, exist_ok=True)

csv_files = {
    "phase1_train_pristine": "phase1_train_pristine_quality_features.csv",
    "phase2_train_pristine": "phase2_train_pristine_quality_features.csv",
    "phase2_eval_pristine": "phase2_eval_pristine_quality_features.csv",
    "phase2_test_pristine": "phase2_test_pristine_quality_features.csv",
    "phase2_train_degraded": "phase2_train_degraded_quality_features.csv",
    "phase2_eval_degraded": "phase2_eval_degraded_quality_features.csv",
    "phase2_test_degraded": "phase2_test_degraded_quality_features.csv"
}

dataframes = {}

for name, filename in csv_files.items():
    path = os.path.join(INPUT, filename)
    dataframes[name] = pd.read_csv(path)
    print(name, ":", dataframes[name].shape)

print("\nAll seven CSV files loaded successfully.")

In [ ]:
severity_tables = []

for name, df in dataframes.items():

    counts = df["severity_label"].value_counts()

    percentages = (
        df["severity_label"].value_counts(normalize=True) * 100
    )

    for severity in ["pristine", "mild", "moderate", "extreme"]:

        severity_tables.append({
            "dataset": name,
            "severity": severity,
            "image_count": counts.get(severity, 0),
            "percentage": round(percentages.get(severity, 0), 2)
        })

severity_summary = pd.DataFrame(severity_tables)

severity_summary.to_csv(
    os.path.join(OUTPUT, "severity_distribution_summary.csv"),
    index=False
)

print("SEVERITY DISTRIBUTION SUMMARY")
print("=" * 70)

for name in csv_files:
    print("\n", name)
    print(
        severity_summary[
            severity_summary["dataset"] == name
        ][["severity", "image_count", "percentage"]].to_string(index=False)
    )

print("\nSummary saved successfully.")

In [ ]:
severity_order = ["pristine", "mild", "moderate", "extreme"]

for name, df in dataframes.items():

    counts = (
        df["severity_label"]
        .value_counts()
        .reindex(severity_order, fill_value=0)
    )

    plt.figure(figsize=(8, 5))

    plt.bar(counts.index, counts.values)

    plt.title(f"Severity Distribution - {name}")
    plt.xlabel("Severity Level")
    plt.ylabel("Number of Images")

    plt.xticks(rotation=0)
    plt.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()

    plot_path = os.path.join(
        OUTPUT,
        f"{name}_severity_distribution.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.show()
    plt.close()

    print("Saved:", plot_path)

print("\nAll seven severity distribution plots generated.")

In [ ]:
feature_columns = [
    "laplacian_variance",
    "median_filter_residual",
    "brightness",
    "contrast",
    "extreme_pixel_percentage",
    "rgb_colour_variation",
    "edge_density"
]

feature_statistics = []

for name, df in dataframes.items():

    grouped = df.groupby("severity_label")[feature_columns].agg(
        ["mean", "std"]
    )

    for severity in ["pristine", "mild", "moderate", "extreme"]:

        if severity in grouped.index:

            for feature in feature_columns:

                feature_statistics.append({
                    "dataset": name,
                    "severity": severity,
                    "feature": feature,
                    "mean": grouped.loc[severity, (feature, "mean")],
                    "standard_deviation": grouped.loc[severity, (feature, "std")]
                })

feature_statistics_df = pd.DataFrame(feature_statistics)

statistics_path = os.path.join(
    OUTPUT,
    "quality_feature_statistics_by_severity.csv"
)

feature_statistics_df.to_csv(statistics_path, index=False)

print("QUALITY FEATURE STATISTICS")
print("=" * 75)

print(feature_statistics_df.to_string(index=False))

print("\nTotal statistical records:", len(feature_statistics_df))
print("Saved:", statistics_path)

In [ ]:
severity_order = ["pristine", "mild", "moderate", "extreme"]

degraded_datasets = [
    "phase2_train_degraded",
    "phase2_eval_degraded",
    "phase2_test_degraded"
]

for name in degraded_datasets:

    df = dataframes[name]

    for feature in feature_columns:

        grouped = df.groupby("severity_label")[feature].agg(
            ["mean", "std"]
        ).reindex(severity_order)

        plt.figure(figsize=(8, 5))

        plt.bar(
            grouped.index,
            grouped["mean"],
            yerr=grouped["std"],
            capsize=4
        )

        plt.title(f"{feature} by Severity - {name}")
        plt.xlabel("Severity Level")
        plt.ylabel(feature.replace("_", " ").title())

        plt.grid(axis="y", linestyle="--", alpha=0.4)
        plt.tight_layout()

        plot_path = os.path.join(
            OUTPUT,
            f"{name}_{feature}_comparison.png"
        )

        plt.savefig(plot_path, dpi=300)
        plt.close()

        print("Saved:", plot_path)

print("\nAll 21 quality-feature comparison plots generated.")

In [ ]:
import glob
import zipfile
import os

png_files = glob.glob(os.path.join(OUTPUT, "*.png"))
csv_files_output = glob.glob(os.path.join(OUTPUT, "*.csv"))

print("Severity and quality analysis files")
print("PNG plots:", len(png_files))
print("CSV summaries:", len(csv_files_output))

assert len(png_files) == 28, "Expected 28 plots"
assert len(csv_files_output) == 2, "Expected 2 CSV summaries"

zip_path = "/content/WBC_Severity_Analysis.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for file in png_files + csv_files_output:
        zipf.write(
            file,
            arcname=os.path.basename(file)
        )

print("\nZIP created successfully:", zip_path)
print("ZIP size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

## Additional comparison plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. LOAD THE SEVEN EXISTING CSV FILES
# --------------------------------------------------

csv_folder = "/content/WBC_Quality_Feature_CSVs"

plot_folder = "/content/WBC_Severity_Analysis/Additional_Comparison_Plots"

os.makedirs(plot_folder, exist_ok=True)

csv_files = {
    "phase1_train_pristine": "phase1_train_pristine_quality_features.csv",
    "phase2_train_pristine": "phase2_train_pristine_quality_features.csv",
    "phase2_eval_pristine": "phase2_eval_pristine_quality_features.csv",
    "phase2_test_pristine": "phase2_test_pristine_quality_features.csv",
    "phase2_train_degraded": "phase2_train_degraded_quality_features.csv",
    "phase2_eval_degraded": "phase2_eval_degraded_quality_features.csv",
    "phase2_test_degraded": "phase2_test_degraded_quality_features.csv"
}

dataframes = {}

for name, filename in csv_files.items():
    dataframes[name] = pd.read_csv(
        os.path.join(csv_folder, filename)
    )

print("All seven CSV files loaded.")

# --------------------------------------------------
# 2. DEFINE THE SEVEN FEATURES
# --------------------------------------------------

features = [
    "laplacian_variance",
    "median_filter_residual",
    "brightness",
    "contrast",
    "extreme_pixel_percentage",
    "rgb_colour_variation",
    "edge_density"
]

severities = ["pristine", "mild", "moderate", "extreme"]

splits = ["train", "eval", "test"]

# Log1p is used only for visualization.
# Original CSV values are not modified.

def plot_values(df, feature):
    values = pd.to_numeric(
        df[feature], errors="coerce"
    ).dropna()

    if len(values) == 0:
        return np.array([])

    if (values < 0).any():
        raise ValueError(
            f"Negative values found in {feature}. "
            "Review transformation before plotting."
        )

    return np.log1p(values.to_numpy())


# --------------------------------------------------
# 3. PHASE 2: PRISTINE SOURCE VS DEGRADED SOURCE
# --------------------------------------------------

count = 0

for split in splits:

    pristine_df = dataframes[
        f"phase2_{split}_pristine"
    ]

    degraded_df = dataframes[
        f"phase2_{split}_degraded"
    ]

    for feature in features:

        groups = [
            plot_values(pristine_df, feature)
        ]

        labels = ["Pristine source"]

        for severity in severities:

            subset = degraded_df[
                degraded_df["severity_label"].str.lower()
                == severity
            ]

            groups.append(plot_values(subset, feature))

            if severity == "pristine":
                labels.append("Degraded source:\npristine")
            else:
                labels.append(severity.capitalize())

        plt.figure(figsize=(11, 6))

        plt.boxplot(
            groups,
            labels=labels,
            showfliers=False
        )

        plt.title(
            f"Phase 2 {split.capitalize()}: "
            f"{feature} by source and severity"
        )

        plt.ylabel("log1p(feature value)")
        plt.xlabel("Source / Severity")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()

        filename = f"{split}_{feature}_source_severity.png"

        plt.savefig(
            os.path.join(plot_folder, filename),
            dpi=200
        )

        plt.close()
        count += 1

print("Source vs severity plots created:", count)


# --------------------------------------------------
# 4. PHASE 1 VS PHASE 2 PRISTINE TRAIN
# --------------------------------------------------

count = 0

phase1 = dataframes["phase1_train_pristine"]
phase2 = dataframes["phase2_train_pristine"]

for feature in features:

    groups = [
        plot_values(phase1, feature),
        plot_values(phase2, feature)
    ]

    plt.figure(figsize=(8, 6))

    plt.boxplot(
        groups,
        labels=["Phase 1 Train", "Phase 2 Train"],
        showfliers=False
    )

    plt.title(
        f"Phase 1 vs Phase 2 Pristine Train: {feature}"
    )

    plt.ylabel("log1p(feature value)")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            plot_folder,
            f"phase1_vs_phase2_{feature}.png"
        ),
        dpi=200
    )

    plt.close()
    count += 1

print("Phase 1 vs Phase 2 plots created:", count)


# --------------------------------------------------
# 5. TRAIN VS EVAL VS TEST FEATURE DISTRIBUTIONS
# --------------------------------------------------

count = 0

for feature in features:

    groups = []
    labels = []

    for split in splits:

        df = dataframes[f"phase2_{split}_degraded"]

        for severity in severities:

            subset = df[
                df["severity_label"].str.lower()
                == severity
            ]

            groups.append(plot_values(subset, feature))

            labels.append(
                f"{split.capitalize()}\n{severity.capitalize()}"
            )

    plt.figure(figsize=(15, 7))

    plt.boxplot(
        groups,
        labels=labels,
        showfliers=False
    )

    plt.title(
        f"Train vs Eval vs Test: {feature} by severity"
    )

    plt.ylabel("log1p(feature value)")
    plt.xlabel("Split and severity")
    plt.xticks(rotation=0)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            plot_folder,
            f"train_eval_test_{feature}.png"
        ),
        dpi=200
    )

    plt.close()
    count += 1

print("Train/Eval/Test comparison plots created:", count)


# --------------------------------------------------
# 6. FINAL PLOT COUNT
# --------------------------------------------------

plot_files = [
    f for f in os.listdir(plot_folder)
    if f.endswith(".png")
]

print("\n======================================")
print("ADDITIONAL PLOT GENERATION COMPLETED")
print("======================================")

print("Source vs severity plots: 21")
print("Phase 1 vs Phase 2 plots: 7")
print("Train/Eval/Test plots: 7")
print("Expected total: 35")
print("Actual PNG files:", len(plot_files))
print("Saved in:", plot_folder)

## Final output packaging

In [ ]:
import os
import zipfile
from pathlib import Path

csv_folder = Path("/content/WBC_Quality_Feature_CSVs")
plot_folder = Path("/content/WBC_Severity_Analysis")

# Verify expected files
feature_csvs = list(csv_folder.glob("*_quality_features.csv"))
all_plots = list(plot_folder.rglob("*.png"))

assert len(feature_csvs) == 7, "Expected exactly 7 feature CSVs"
assert len(all_plots) == 63, "Expected exactly 63 plots"

archives = {
    "/content/WBC_Quality_Feature_CSVs.zip": [
        (csv_folder, "WBC_Quality_Feature_CSVs")
    ],
    "/content/WBC_Severity_Analysis.zip": [
        (plot_folder, "WBC_Severity_Analysis")
    ],
    "/content/WBC_Project_Handover.zip": [
        (csv_folder, "WBC_Quality_Feature_CSVs"),
        (plot_folder, "WBC_Severity_Analysis")
    ]
}

for archive_path, folders in archives.items():

    with zipfile.ZipFile(
        archive_path, "w", zipfile.ZIP_DEFLATED
    ) as zipf:

        for folder, archive_name in folders:

            for file in folder.rglob("*"):

                if file.is_file():
                    relative_path = file.relative_to(folder)
                    archive_name_path = (
                        Path(archive_name) / relative_path
                    )

                    zipf.write(file, archive_name_path)

    print(
        os.path.basename(archive_path),
        "| Size:",
        round(os.path.getsize(archive_path) / (1024 * 1024), 2),
        "MB"
    )

print("\nAll three ZIP archives created successfully.")